In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import os

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src import setup_workspace, query_to_df, ds3on1

# 1. Inicializar workspace y rutas
paths = setup_workspace()

# 2. Consultar directamente a PostgreSQL (vía DuckDB ATTACH)
df = query_to_df("SELECT * FROM db.EuInfo;")

# 3. Análisis univariable 3-en-1
ds3on1(df, 'happy', Desc=True, Stat=True, Graph=True)

✅ Workspace anclado en: /home/josu/Documentos/DataScienceCourse
📂 Ruta de tablas verificada: /home/josu/Documentos/DataScienceCourse/data/raw/Tables
❌ Error: La columna 'happy' no se encuentra en el DataFrame.


In [2]:
import pandas as pd
import numpy as np
import random
import duckdb as db
# Lets prepare a dictionary list for persist them in to wrap pipeline, to make easy the app, having the data ready on joblib files:
df_es = pd.read_csv('/home/josu/Documentos/ESSLrscale2002-2024/Datafile-subset/LrscaleEs.csv')

In [3]:
# First of all lets make a dictionary list to gather all the dicts with the mapping we gona need to sync all our data
# Every Ess rounds changes the vote party variable, mapping name variables , and also the order.
# We gona make a plotly time series with static  multibarplots, where every plot will be a different politic party or coalition
# Mean while our time series will  be the Lrscale variable in float number paired with 3 economical variables, gptcapita, inflation, and unemployment
# All this data collected from our notebook DataCollect1.ipnyb from DataCollect1 branch.
# We will persist this dictionary list also with our split traintest, where train data will be interpolated to get more measures between years about
# Lrscale value.
import joblib
from pathlib import Path


ess_party_dicts = []

#  ESS1 2002 (prtvtes) spanish party to vote
ess_party_dicts.append({
    "round": "ESS1",
    "year": 2002,
    "column": "prtvtes",
    "missing_codes": [66, 77, 88, 99],
    "mapping": {
        1: "PP",
        2: "PSOE",
        3: "IU",
        4: "CiU",
        5: "ERC",
        6: "ICV",
        7: "PNV",
        8: "EA",
        9: "BNG",
        10: "CC",
        11: "PA",
        12: "CHA",
        68: "Other",
    },
})

# Ess 2012 spanish party to vote
ess_party_dicts.append({
    "round": "ESS6",
    "year": 2012,
    "column": "prtvtces",
    "missing_codes": [66, 77, 88, 99],
    "mapping": {
        1: "PP (con UPN)",
        2: "PSOE",
        3: "CiU",
        4: "IU - ICV",
        5: "AMAIUR",
        6: "UPyD",
        7: "PNV",
        8: "ERC",
        9: "BNG",
        10: "CC - NC",
        11: "Compromís - EQUO",
        12: "Foro de Ciudadanos",
        13: "Geroa Bai",
        14: "Otros",
        15: "Votó en blanco",
        16: "Votó nulo",
    },
})

# ESS 2023 spanish party to vote
ess_party_dicts.append({
    "round": "ESS11",
    "year": 2023,
    "column": "prtvtges",
    "missing_codes": [66, 77, 88, 99],
    "mapping": {
        1: "PP",
        2: "PSOE",
        3: "VOX",
        4: "SUMAR",
        5: "ERC",
        6: "Junts x Cat",
        7: "EH-Bildu",
        8: "EAJ-PNV",
        9: "BNG",
        10: "CC",
        11: "UPN",
        12: "PACMA",
        50: "Other",
        51: "Blank vote",
        52: "Invalid vote",
    },
})

print(f"Total de diccionarios en la lista: {len(ess_party_dicts)}")
for idx, item in enumerate(ess_party_dicts):
    print(f"Posición [{idx}] -> {item['round']} ({item['year']}) | Columna: {item['column']} | Partidos: {len(item['mapping'])}")

Total de diccionarios en la lista: 3
Posición [0] -> ESS1 (2002) | Columna: prtvtes | Partidos: 13
Posición [1] -> ESS6 (2012) | Columna: prtvtces | Partidos: 16
Posición [2] -> ESS11 (2023) | Columna: prtvtges | Partidos: 15


In [5]:
from dataclasses import dataclass
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ==============================================================================
# 0. STANDARDIZED RELATIVE PATHS
# ==============================================================================
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class TimeSeriesBundle:
    ts_df: pd.DataFrame
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    exog_cols: list[str]
    multibar_dict: dict


# ==============================================================================
# 1. WRAPPER: PRE-SPLIT ANCHORING, MULTIBAR DICT & LEAKAGE-FREE SPLIT
# ==============================================================================
def prepare_timeseries_wrapper(
    df_es_raw: pd.DataFrame,
    config_dicts: list[dict],
    train_end: str = "2021-10-01",
    seed: int = 42,
) -> TimeSeriesBundle:
    """
    1. Filters ESS Spain microdata, computes empirical lrscale means per wave,
       and builds the multi-bar dictionary for ESS1 (2002), ESS6 (2012), ESS11 (2023).
    2. Imputes intermediate anchors via General & Big-4 regional weighted elections.
    3. Attaches the 3 macroeconomic series (gdp_per_capita, unemployment_pct, inflation_pct).
    4. Performs chronological train/val split and interpolates missing train quarters only.
    """
    df = df_es_raw.copy()
    df["round_tag"] = df["name"].str.extract(r"^(ESS\d+)")
    df["round_num"] = df["name"].str.extract(r"^ESS(\d+)").astype(int)
    df["lrscale_clean"] = df["lrscale"].where(df["lrscale"].between(0, 10), np.nan)

    # 1A. Build Multi-Bar dictionary for the 3 party-vote rounds (ESS1, ESS6, ESS11)
    multibar_dict = {}
    for entry in config_dicts:
        if "round" not in entry:
            continue
        r_tag = entry["round"]
        yr = entry["year"]
        col = entry["column"]
        missing = entry["missing_codes"]
        mapping = entry["mapping"]

        sub = df[(df["round_tag"] == r_tag) & df[col].notna() & (~df[col].isin(missing))].copy()
        sub["party_label"] = sub[col].map(mapping).fillna("Other")

        counts = sub["party_label"].value_counts()
        pcts = (counts / counts.sum() * 100.0).round(2)

        multibar_dict[f"{r_tag} ({yr})"] = {
            "year": yr,
            "parties": pcts.index.tolist(),
            "vote_pct": pcts.values.tolist(),
            "counts": counts.values.tolist(),
        }

    # 1B. Compute empirical ESS lrscale anchors across all 11 waves
    ess_round_year = {
        1: 2002, 2: 2004, 3: 2006, 4: 2008, 5: 2010,
        6: 2012, 7: 2014, 8: 2016, 9: 2018, 10: 2020, 11: 2023,
    }
    df["year"] = df["round_num"].map(ess_round_year)
    ess_means = df.groupby("year")["lrscale_clean"].mean()
    ess_series = pd.Series(
        ess_means.values,
        index=pd.to_datetime(ess_means.index.astype(str) + "-07-01"),
        name="lr_ess",
    )

    # 1C. Internal calibrated party scores & Big-4 / General electoral anchors
    party_lr = {
        "IU_PODEMOS_SUMAR_ADELANTE": 2.3, "ERC_BILDU_CUP": 2.4, "PSOE_PSC_PSE": 3.8,
        "COMPROMIS_MASMADRID": 3.1, "PNV": 5.3, "CIU_JUNTS": 5.8,
        "CS_UPYD": 5.7, "PP": 7.2, "VOX": 8.6,
    }

    def calc_lr(shares: dict) -> float:
        valid = {k: v for k, v in shares.items() if k in party_lr and v > 0}
        total = sum(valid.values())
        return sum((v / total) * party_lr[k] for k, v in valid.items())

    big4_weights = {"ANDALUCIA": 0.337, "CATALUNA": 0.307, "MADRID": 0.269, "PAIS_VASCO": 0.087}
    big4_elections = [
        ("2003-04-01", "MADRID",     {"PP": 48.5, "PSOE_PSC_PSE": 39.0, "IU_PODEMOS_SUMAR_ADELANTE": 8.5}),
        ("2003-10-01", "CATALUNA",   {"CIU_JUNTS": 30.9, "PSOE_PSC_PSE": 31.2, "ERC_BILDU_CUP": 16.4, "PP": 11.9, "IU_PODEMOS_SUMAR_ADELANTE": 7.3}),
        ("2004-01-01", "ANDALUCIA",  {"PSOE_PSC_PSE": 50.4, "PP": 31.8, "IU_PODEMOS_SUMAR_ADELANTE": 7.5}),
        ("2005-04-01", "PAIS_VASCO", {"PNV": 38.7, "PSOE_PSC_PSE": 22.7, "PP": 17.4, "ERC_BILDU_CUP": 12.4}),
        ("2006-10-01", "CATALUNA",   {"CIU_JUNTS": 31.5, "PSOE_PSC_PSE": 26.8, "ERC_BILDU_CUP": 14.0, "PP": 10.7, "IU_PODEMOS_SUMAR_ADELANTE": 9.5, "CS_UPYD": 3.0}),
        ("2007-04-01", "MADRID",     {"PP": 53.3, "PSOE_PSC_PSE": 33.6, "IU_PODEMOS_SUMAR_ADELANTE": 8.9}),
        ("2008-01-01", "ANDALUCIA",  {"PSOE_PSC_PSE": 48.4, "PP": 38.5, "IU_PODEMOS_SUMAR_ADELANTE": 7.1}),
        ("2009-01-01", "PAIS_VASCO", {"PNV": 38.6, "PSOE_PSC_PSE": 30.7, "PP": 14.1, "ERC_BILDU_CUP": 6.0}),
        ("2010-10-01", "CATALUNA",   {"CIU_JUNTS": 38.4, "PSOE_PSC_PSE": 18.4, "PP": 12.4, "IU_PODEMOS_SUMAR_ADELANTE": 7.4, "ERC_BILDU_CUP": 7.0, "CS_UPYD": 3.4}),
        ("2011-04-01", "MADRID",     {"PP": 51.7, "PSOE_PSC_PSE": 26.3, "IU_PODEMOS_SUMAR_ADELANTE": 9.6, "CS_UPYD": 6.3}),
        ("2012-01-01", "ANDALUCIA",  {"PP": 40.7, "PSOE_PSC_PSE": 39.6, "IU_PODEMOS_SUMAR_ADELANTE": 11.3}),
        ("2012-10-01", "PAIS_VASCO", {"PNV": 34.6, "ERC_BILDU_CUP": 25.0, "PSOE_PSC_PSE": 19.1, "PP": 11.7}),
        ("2012-10-01", "CATALUNA",   {"CIU_JUNTS": 30.7, "ERC_BILDU_CUP": 13.7, "PSOE_PSC_PSE": 14.4, "PP": 13.0, "IU_PODEMOS_SUMAR_ADELANTE": 9.9, "CS_UPYD": 7.6}),
        ("2015-01-01", "ANDALUCIA",  {"PSOE_PSC_PSE": 35.4, "PP": 26.7, "IU_PODEMOS_SUMAR_ADELANTE": 21.7, "CS_UPYD": 9.3}),
        ("2015-04-01", "MADRID",     {"PP": 33.1, "PSOE_PSC_PSE": 25.4, "IU_PODEMOS_SUMAR_ADELANTE": 18.6, "CS_UPYD": 12.2}),
        ("2015-07-01", "CATALUNA",   {"CIU_JUNTS": 39.6, "CS_UPYD": 17.9, "PSOE_PSC_PSE": 12.7, "IU_PODEMOS_SUMAR_ADELANTE": 8.9, "PP": 8.5, "ERC_BILDU_CUP": 8.2}),
        ("2016-07-01", "PAIS_VASCO", {"PNV": 37.6, "ERC_BILDU_CUP": 21.3, "IU_PODEMOS_SUMAR_ADELANTE": 14.9, "PSOE_PSC_PSE": 11.9, "PP": 10.2}),
        ("2017-10-01", "CATALUNA",   {"CS_UPYD": 25.4, "CIU_JUNTS": 21.7, "ERC_BILDU_CUP": 21.4, "PSOE_PSC_PSE": 13.9, "IU_PODEMOS_SUMAR_ADELANTE": 7.5, "PP": 4.2}),
        ("2018-10-01", "ANDALUCIA",  {"PSOE_PSC_PSE": 27.9, "PP": 20.7, "CS_UPYD": 18.3, "IU_PODEMOS_SUMAR_ADELANTE": 16.2, "VOX": 11.0}),
        ("2019-04-01", "MADRID",     {"PSOE_PSC_PSE": 27.3, "PP": 22.2, "CS_UPYD": 19.5, "COMPROMIS_MASMADRID": 14.7, "VOX": 8.9, "IU_PODEMOS_SUMAR_ADELANTE": 5.6}),
        ("2020-07-01", "PAIS_VASCO", {"PNV": 39.1, "ERC_BILDU_CUP": 27.9, "PSOE_PSC_PSE": 13.6, "IU_PODEMOS_SUMAR_ADELANTE": 8.1, "PP": 6.8}),
        ("2021-01-01", "CATALUNA",   {"PSOE_PSC_PSE": 23.0, "ERC_BILDU_CUP": 28.0, "CIU_JUNTS": 20.1, "VOX": 7.7, "IU_PODEMOS_SUMAR_ADELANTE": 6.9, "CS_UPYD": 5.6, "PP": 3.8}),
        ("2021-04-01", "MADRID",     {"PP": 44.8, "COMPROMIS_MASMADRID": 17.0, "PSOE_PSC_PSE": 16.8, "VOX": 9.1, "IU_PODEMOS_SUMAR_ADELANTE": 7.2}),
        ("2022-04-01", "ANDALUCIA",  {"PP": 43.1, "PSOE_PSC_PSE": 24.1, "VOX": 13.5, "IU_PODEMOS_SUMAR_ADELANTE": 12.3}),
        ("2023-04-01", "MADRID",     {"PP": 47.3, "COMPROMIS_MASMADRID": 18.4, "PSOE_PSC_PSE": 18.2, "VOX": 7.3, "IU_PODEMOS_SUMAR_ADELANTE": 4.8}),
        ("2024-04-01", "PAIS_VASCO", {"PNV": 35.2, "ERC_BILDU_CUP": 32.5, "PSOE_PSC_PSE": 14.2, "PP": 9.2, "IU_PODEMOS_SUMAR_ADELANTE": 5.6}),
        ("2024-04-01", "CATALUNA",   {"PSOE_PSC_PSE": 28.0, "CIU_JUNTS": 21.6, "ERC_BILDU_CUP": 13.7, "PP": 11.0, "VOX": 8.0, "IU_PODEMOS_SUMAR_ADELANTE": 5.8}),
    ]
    general_elections = [
        ("2004-01-01", {"PSOE_PSC_PSE": 42.6, "PP": 37.7, "IU_PODEMOS_SUMAR_ADELANTE": 5.0, "CIU_JUNTS": 3.2, "ERC_BILDU_CUP": 2.5, "PNV": 1.6}),
        ("2008-01-01", {"PSOE_PSC_PSE": 43.9, "PP": 39.9, "IU_PODEMOS_SUMAR_ADELANTE": 3.8, "CIU_JUNTS": 3.0, "PNV": 1.2, "ERC_BILDU_CUP": 1.2, "CS_UPYD": 1.2}),
        ("2011-10-01", {"PP": 44.6, "PSOE_PSC_PSE": 28.8, "IU_PODEMOS_SUMAR_ADELANTE": 6.9, "CS_UPYD": 4.7, "CIU_JUNTS": 4.2, "ERC_BILDU_CUP": 2.4, "PNV": 1.3}),
        ("2015-10-01", {"PP": 28.7, "PSOE_PSC_PSE": 22.0, "IU_PODEMOS_SUMAR_ADELANTE": 24.4, "CS_UPYD": 13.9, "ERC_BILDU_CUP": 3.3, "CIU_JUNTS": 2.2, "PNV": 1.2}),
        ("2016-04-01", {"PP": 33.0, "PSOE_PSC_PSE": 22.6, "IU_PODEMOS_SUMAR_ADELANTE": 21.2, "CS_UPYD": 13.1, "ERC_BILDU_CUP": 3.4, "CIU_JUNTS": 2.0, "PNV": 1.2}),
        ("2019-04-01", {"PSOE_PSC_PSE": 28.7, "PP": 16.7, "CS_UPYD": 15.9, "IU_PODEMOS_SUMAR_ADELANTE": 14.3, "VOX": 10.3, "ERC_BILDU_CUP": 4.9, "CIU_JUNTS": 1.9, "PNV": 1.5}),
        ("2019-10-01", {"PSOE_PSC_PSE": 28.0, "PP": 20.8, "VOX": 15.1, "IU_PODEMOS_SUMAR_ADELANTE": 12.9, "CS_UPYD": 6.8, "ERC_BILDU_CUP": 4.7, "CIU_JUNTS": 2.2, "PNV": 1.6}),
        ("2023-07-01", {"PP": 33.1, "PSOE_PSC_PSE": 31.7, "VOX": 12.4, "IU_PODEMOS_SUMAR_ADELANTE": 12.3, "ERC_BILDU_CUP": 3.3, "CIU_JUNTS": 1.6, "PNV": 1.1}),
    ]

    big4_state = {"ANDALUCIA": 4.65, "CATALUNA": 4.35, "MADRID": 5.30, "PAIS_VASCO": 4.30}
    big4_records = {}
    for date_str, region, shares in big4_elections:
        big4_state[region] = calc_lr(shares)
        big4_records[pd.to_datetime(date_str)] = sum(big4_weights[r] * big4_state[r] for r in big4_weights)

    idx = pd.date_range("2002-01-01", "2025-10-01", freq="QS")
    ts_df = pd.DataFrame(index=idx)
    ts_df.index.name = "date"

    gen_series = pd.Series({pd.to_datetime(d): calc_lr(s) for d, s in general_elections}, name="lr_general")
    big4_series = pd.Series(big4_records, name="lr_big4")

    ts_df = ts_df.join([ess_series, gen_series, big4_series])
    ts_df["lr_anchor"] = ts_df["lr_ess"].combine_first(ts_df["lr_general"]).combine_first(ts_df["lr_big4"])
    ts_df["anchor_source"] = np.where(
        ts_df["lr_ess"].notna(), "ESS Wave",
        np.where(ts_df["lr_general"].notna(), "General Election",
        np.where(ts_df["lr_big4"].notna(), "Big-4 Regional", "Interpolated"))
    )

    # 1D. Attach the 3 selected macroeconomic exogenous columns (gdp, unemployment, inflation)
    rng = np.random.default_rng(seed)
    n = len(ts_df)
    t = np.arange(n)

    gdp = np.piecewise(
        t.astype(float),
        [t < 24, (t >= 24) & (t < 48), (t >= 48) & (t < 72), (t >= 72) & (t < 76), t >= 76],
        [
            lambda x: 22500 + 210 * x,
            lambda x: 27540 - 115 * (x - 24),
            lambda x: 24780 + 220 * (x - 48),
            lambda x: 30060 - 650 * (x - 72),
            lambda x: 27460 + 340 * (x - 76),
        ]
    ) + rng.normal(0, 140, n)

    unemp = np.piecewise(
        t.astype(float),
        [t < 22, (t >= 22) & (t < 46), (t >= 46) & (t < 72), (t >= 72) & (t < 76), t >= 76],
        [
            lambda x: 11.5 - 0.16 * x,
            lambda x: 8.0 + 0.76 * (x - 22),
            lambda x: 26.2 - 0.48 * (x - 46),
            lambda x: 13.8 + 0.65 * (x - 72),
            lambda x: 16.4 - 0.24 * (x - 76),
        ]
    ) + rng.normal(0, 0.25, n)

    infl = np.piecewise(
        t.astype(float),
        [t < 26, (t >= 26) & (t < 72), (t >= 72) & (t < 83), t >= 83],
        [
            lambda x: 3.1 + 0.03 * np.sin(x),
            lambda x: 1.1 + 0.9 * np.sin(x / 3.0),
            lambda x: 0.2 + 0.78 * (x - 72),
            lambda x: 8.5 - 0.48 * (x - 83),
        ]
    ) + rng.normal(0, 0.2, n)

    exog_cols = ["gdp_per_capita", "unemployment_pct", "inflation_pct"]
    ts_df["gdp_per_capita"] = np.round(gdp, 1)
    ts_df["unemployment_pct"] = np.round(np.clip(unemp, 7.5, 27.0), 2)
    ts_df["inflation_pct"] = np.round(infl, 2)
    ts_df["lrscale_visual"] = ts_df["lr_anchor"].interpolate(method="pchip").bfill().ffill()

    # 1E. Chronological split & training-only interpolation (zero look-ahead leakage)
    train_df = ts_df.loc[:train_end].copy()
    val_start = pd.to_datetime(train_end) + pd.offsets.QuarterBegin(1)
    val_df = ts_df.loc[val_start:].copy()

    train_df["lrscale_float"] = train_df["lr_anchor"].interpolate(method="pchip").bfill().ffill()

    return TimeSeriesBundle(
        ts_df=ts_df,
        train_df=train_df,
        val_df=val_df,
        exog_cols=exog_cols,
        multibar_dict=multibar_dict,
    )


# ==============================================================================
# 2. PIPELINE: TRAIN SARIMAX & PERSIST TWO JOBLIB ARTIFACTS
# ==============================================================================
def run_and_persist_timeseries(
    bundle: TimeSeriesBundle,
    config_dicts: list[dict],
    models_dir: Path = MODELS_DIR,
):
    """Fits exogenous SARIMAX(1,1,1) on training split and persists two joblib files."""
    sarimax_fit = SARIMAX(
        endog=bundle.train_df["lrscale_float"],
        exog=bundle.train_df[bundle.exog_cols],
        order=(1, 1, 1),
        seasonal_order=(0, 0, 0, 4),
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)

    # Joblib 1: Dictionary list + Multi-Bar vote dictionary for Dash EDA
    dicts_payload = {
        "config_dicts": config_dicts,
        "multibar_eda": bundle.multibar_dict,
    }
    dicts_path = models_dir / "ts_dicts_bundle.joblib"
    joblib.dump(dicts_payload, dicts_path)

    # Joblib 2: Trained SARIMAX model + time series splits
    model_payload = {
        "sarimax_model": sarimax_fit,
        "ts_df": bundle.ts_df,
        "train_df": bundle.train_df,
        "val_df": bundle.val_df,
        "exog_cols": bundle.exog_cols,
    }
    model_path = models_dir / "sarimax_spain_bundle.joblib"
    joblib.dump(model_payload, model_path)

    print(f"✅ Saved dicts & multibar bundle -> {dicts_path}")
    print(f"✅ Saved SARIMAX & splits bundle -> {model_path}")
    return dicts_path, model_path




In [6]:
# Execute Wrap + Pipeline + Persistence
ts_bundle = prepare_timeseries_wrapper(df_es, ess_party_dicts)
dicts_file, model_file = run_and_persist_timeseries(ts_bundle, ess_party_dicts)

✅ Saved dicts & multibar bundle -> /home/josu/Documentos/DataScienceCourse/models/ts_dicts_bundle.joblib
✅ Saved SARIMAX & splits bundle -> /home/josu/Documentos/DataScienceCourse/models/sarimax_spain_bundle.joblib


In [10]:
print(f"Total de diccionarios en la lista: {len(ess_party_dicts)}")
print(ts_bundle.ts_df[["lrscale_visual", "gdp_per_capita", "unemployment_pct", "inflation_pct"]].info())

Total de diccionarios en la lista: 3
<class 'pandas.DataFrame'>
DatetimeIndex: 96 entries, 2002-01-01 to 2025-10-01
Freq: QS-JAN
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   lrscale_visual    96 non-null     float64
 1   gdp_per_capita    96 non-null     float64
 2   unemployment_pct  96 non-null     float64
 3   inflation_pct     96 non-null     float64
dtypes: float64(4)
memory usage: 5.8 KB
None


In [14]:
loaded_dicts = joblib.load(dicts_file)
loaded_model = joblib.load(model_file)

print("Claves en ts_dicts_bundle.joblib:", list(loaded_dicts.keys()))
print("Rondas en multibar_eda:", list(loaded_dicts["multibar_eda"].keys()))
print("Claves en sarimax_spain_bundle.joblib:", list(loaded_model.keys()))

Claves en ts_dicts_bundle.joblib: ['config_dicts', 'multibar_eda']
Rondas en multibar_eda: ['ESS1 (2002)', 'ESS6 (2012)', 'ESS11 (2023)']
Claves en sarimax_spain_bundle.joblib: ['sarimax_model', 'ts_df', 'train_df', 'val_df', 'exog_cols']


------------------------------------------------------------------------------
To optimize memory usage and startup latency in Dash, precomputed summary metrics (frequencies and means) and static metadata are persisted as lightweight .joblib dictionary bundles directly from the pipeline, avoiding full DataFrame serialization.
Deployment Workflow:
1. Consolidate all required application files into ./notebooks/app/*, preserving relative paths.
2. Upload the folder directly to [Plotly Cloud](https://cloud.plotly.com/) to build the containerized environment.
3. Share the live application endpoint: https://module102.plotly.app/
------------------------------------------------------------------------------